# ChatGPT решения

### Задача 1:
Классическая головоломка. У вас имеется 100 долларов, и вы можете купить 100 животных. Собаки стоят по 15 долларов, коты по 1 доллару, мыши по 25 центов. Вы должны купить хотя бы по одному животному каждого вида. Найдите решение этого паззла.


In [3]:
from z3 import *

# Переменные
dogs = Int("dogs")
cats = Int("cats")
mice = Int("mice")

s = Solver()

# Все количества положительные
s.add(dogs >= 1)
s.add(cats >= 1)
s.add(mice >= 1)

# Всего 100 животных
s.add(dogs + cats + mice == 100)

# Стоимость:
# Собака = 15$, кот = 1$, мышь = 0.25$
# Умножаем на 4, чтобы избежать дробей.
s.add(60 * dogs + 4 * cats + mice == 400)

if s.check() == sat:
    model = s.model()
    print(f"dogs = {model[dogs]}")
    print(f"cats  = {model[cats]}")
    print(f"mice  = {model[mice]}")
else:
    print("No solution")

dogs = 3
cats  = 41
mice  = 56


Если хочется убедиться, что решение единственное, можно после нахождения модели запретить её и проверить существование другой:

In [4]:
m = s.model()

d = m[dogs].as_long()
c = m[cats].as_long()
mi = m[mice].as_long()

# Запрещаем найденную модель
s.add(Or(
    dogs != d,
    cats != c,
    mice != mi
))

print(s.check())  # unsat

unsat


### Задача 2:

Судоку -- это классическая и очень популярная головоломка, которая стала самым популярным паззлом при обучении солверам. Есть квадрат 9x9 клеток, и надо заполнить его цифрами от 1 до 9 так, чтобы в каждом столбце, в каждой строке и в каждом квадрате 3x3 клетки содержались все цифры от 1 до 9.

In [5]:
from z3 import *

# Размер поля
N = 9

# Создаем переменные
cells = [[Int(f"x_{i}_{j}") for j in range(N)] for i in range(N)]

solver = Solver()

# Каждая клетка содержит число от 1 до 9
for i in range(N):
    for j in range(N):
        solver.add(And(cells[i][j] >= 1, cells[i][j] <= 9))

# Все числа в каждой строке различны
for i in range(N):
    solver.add(Distinct(cells[i]))

# Все числа в каждом столбце различны
for j in range(N):
    solver.add(Distinct([cells[i][j] for i in range(N)]))

# Все числа в каждом квадрате 3x3 различны
for bi in range(3):
    for bj in range(3):
        block = [
            cells[i][j]
            for i in range(bi * 3, bi * 3 + 3)
            for j in range(bj * 3, bj * 3 + 3)
        ]
        solver.add(Distinct(block))

# Исходная головоломка (0 означает пустую клетку)
puzzle = [
    [5,3,0,0,7,0,0,0,0],
    [6,0,0,1,9,5,0,0,0],
    [0,9,8,0,0,0,0,6,0],
    [8,0,0,0,6,0,0,0,3],
    [4,0,0,8,0,3,0,0,1],
    [7,0,0,0,2,0,0,0,6],
    [0,6,0,0,0,0,2,8,0],
    [0,0,0,4,1,9,0,0,5],
    [0,0,0,0,8,0,0,7,9]
]

# Добавляем известные значения
for i in range(N):
    for j in range(N):
        if puzzle[i][j] != 0:
            solver.add(cells[i][j] == puzzle[i][j])

# Решаем
if solver.check() == sat:
    model = solver.model()
    solution = [
        [model.evaluate(cells[i][j]).as_long() for j in range(N)]
        for i in range(N)
    ]

    for row in solution:
        print(*row)
else:
    print("Решение не существует")

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


Основные ограничения

Диапазон значений

1 <= x[i][j] <= 9

Строки

Distinct(cells[i])

Столбцы

Distinct([cells[i][j] for i in range(9)])

Квадраты 3×3

Distinct(block)

Фиксированные клетки

solver.add(cells[i][j] == puzzle[i][j])

### Задача 3:

На шахматной доске надо расставить восемь ферзей так, чтобы никто друг друга не атаковал. В терминах условий Z3, нам по сути требуется задать всего одно ограничение: чтобы никакие два ферзя не располагались в одном столбце, строке или диагонали.

In [6]:
from z3 import *

N = 8

# q[i] — столбец ферзя в i-й строке
q = [Int(f"q_{i}") for i in range(N)]

s = Solver()

# Ферзи находятся внутри доски
for i in range(N):
    s.add(q[i] >= 0, q[i] < N)

# Все столбцы различны
s.add(Distinct(q))

# Нет двух ферзей на одной диагонали
for i in range(N):
    for j in range(i + 1, N):
        s.add(Abs(q[i] - q[j]) != j - i)

if s.check() == sat:
    m = s.model()

    board = [["." for _ in range(N)] for _ in range(N)]
    for row in range(N):
        col = m[q[row]].as_long()
        board[row][col] = "Q"

    for row in board:
        print(" ".join(row))
else:
    print("Решения нет.")

. . . . . . Q .
. . Q . . . . .
. . . . . . . Q
. Q . . . . . .
. . . . Q . . .
Q . . . . . . .
. . . . . Q . .
. . . Q . . . .


### Задача 4:

Один из классических типов головоломок -- это нахождение цифр, соответствующих буквам словесного выражения. Например, имеется выражение send + more = money, и надо вместо букв подставить подходящие цифры (одинаковые буквы -- это одинаковые цифры), чтобы оно получилось истинным.

In [7]:
from z3 import *

# Переменные
S, E, N, D, M, O, R, Y = Ints("S E N D M O R Y")

solver = Solver()

letters = [S, E, N, D, M, O, R, Y]

# Все буквы - цифры
for x in letters:
    solver.add(x >= 0, x <= 9)

# Все цифры различны
solver.add(Distinct(letters))

# Старшие цифры не могут быть нулем
solver.add(S != 0)
solver.add(M != 0)

# Числа
SEND  = 1000*S + 100*E + 10*N + D
MORE  = 1000*M + 100*O + 10*R + E
MONEY = 10000*M + 1000*O + 100*N + 10*E + Y

# Основное ограничение
solver.add(SEND + MORE == MONEY)

if solver.check() == sat:
    model = solver.model()

    for v in letters:
        print(f"{v} = {model[v]}")

    send = model.eval(SEND)
    more = model.eval(MORE)
    money = model.eval(MONEY)

    print(f"\n{send} + {more} = {money}")
else:
    print("Решение не найдено")

S = 9
E = 5
N = 6
D = 7
M = 1
O = 0
R = 8
Y = 2

9567 + 1085 = 10652


Более "z3-стильный" вариант с переносами

In [8]:
from z3 import *

S, E, N, D, M, O, R, Y = Ints("S E N D M O R Y")
c1, c2, c3, c4 = Ints("c1 c2 c3 c4")

solver = Solver()

digits = [S, E, N, D, M, O, R, Y]

solver.add([And(0 <= x, x <= 9) for x in digits])
solver.add(Distinct(digits))
solver.add(S != 0, M != 0)

# переносы могут быть только 0 или 1
solver.add([Or(c == 0, c == 1) for c in [c1, c2, c3, c4]])

# D + E = Y + 10*c1
solver.add(D + E == Y + 10*c1)

# N + R + c1 = E + 10*c2
solver.add(N + R + c1 == E + 10*c2)

# E + O + c2 = N + 10*c3
solver.add(E + O + c2 == N + 10*c3)

# S + M + c3 = O + 10*c4
solver.add(S + M + c3 == O + 10*c4)

# Последний перенос образует старший разряд
solver.add(c4 == M)

print(solver.check())
print(solver.model())

sat
[N = 6,
 M = 1,
 c2 = 1,
 Y = 2,
 S = 9,
 R = 8,
 O = 0,
 c4 = 1,
 c3 = 0,
 D = 7,
 c1 = 1,
 E = 5]


### Сравнение:

На данный момент мне тяжело различить качество решения задач. Как задачи из курса так и варианты GPT рабочие и решаются. В плане легкости восприятия задача про головолокму на курсе решается заметно легче и понятнее чем предложил GPT. По задаче с ннахождением цифр соответствующих буквам словесного выражения GPT предложил 2 варианта - схожий с решением курса и более "стилизованный" для Z3, который мне показался не очень понятным, пришлось вникнуть в его решение. Остальные задачи сходятся в сути решений, однако местами отличаются по используемому синтаксису и способу записи ограничений. 